In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor
from PIL import Image
from torchvision import transforms

# Load tokenizer
path = "OpenGVLab/InternVL2_5-1B"
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)

# Load model with LoRA
checkpoint_path = "/DISK/QuantumVLM/InternVL/internvl_chat/work_dirs/internvl_chat_v2_5/internvl2_5_1b_dynamic_res_2nd_finetune_lora/checkpoint-689"
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(path, trust_remote_code=True)

# Load and preprocess image
image_path = "wigner_3d_alpha.png"
image = Image.open(image_path).convert("RGB")

# Use InternVL's preprocess method (check model API — this may differ)
preprocess = model.preprocess_transform if hasattr(model, "preprocess_transform") else transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])
pixel_values = preprocess(image).unsqueeze(0).to(dtype=torch.bfloat16, device="cuda")

# Prepare text prompt
text_prompt = "<Image>\nWhat is in this image?"
inputs = tokenizer(text_prompt, return_tensors="pt").to("cuda")

print(inputs)

# Generate response
    # Prepare the model inputs
inputs = processor(
    text_prompt,
    images=image,
    return_tensors="pt"
).to("cuda")

# Generation parameters
generation_args = {
    "max_new_tokens": 2000,
    "temperature": 0.3,
    "do_sample": False
}

# Generate the output
generate_ids = model.generate(
    **inputs,
    eos_token_id=processor.tokenizer.eos_token_id,
    **generation_args
)

# Remove input tokens from the output
generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
response = processor.batch_decode(
    generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(response)

/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Some weights of the model checkpoint at /DISK/QuantumVLM/InternVL/internvl_chat/work_dirs/internvl_chat_v2_5/internvl2_5_1b_dynamic_res_2nd_finetune_lora/checkpoint-689 were not used when initializing InternVLChatModel: ['language_model.base_model.model.lm_head.weight', 'language_model.base_model.model.model.embed_tokens.weight', 'language_model.base_model.model.model.layers.0.input_layernorm.weight', 'language_model.base_model.model.model.layers.0.mlp.down_proj.base_layer.weight', 'language_model.base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight', 'language_model.base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight', 'language_model.base_model.model.mode

{'input_ids': tensor([[46465,   397,  3838,   374,   304,   419,  2168,    30]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


TypeError: PreTrainedTokenizerFast._batch_encode_plus() got an unexpected keyword argument 'images'

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Path to your checkpoint folder
checkpoint_path = "/DISK/QuantumVLM/InternVL/internvl_chat/work_dirs/internvl_chat_v2_5/internvl2_5_1b_dynamic_res_2nd_finetune_lora/checkpoint-689"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, trust_remote_code=True)

# Load model from checkpoint
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_path,
    torch_dtype=torch.bfloat16,  # or torch.float16 if that's what you used
    trust_remote_code=True
).cuda().eval()

# Inference
prompt = "Describe the properties of a quantum coherent state."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100)

# Decode and print result
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Some weights of the model checkpoint at /DISK/QuantumVLM/InternVL/internvl_chat/work_dirs/internvl_chat_v2_5/internvl2_5_1b_dynamic_res_2nd_finetune_lora/checkpoint-689 were not used when initializing InternVLChatModel: ['language_model.base_model.model.lm_head.weight', 'language_model.base_model.model.model.embed_tokens.weight', 'language_model.base_model.model.model.layers.0.input_layernorm.weight', 'language_model.b

AssertionError: 

In [3]:
import numpy as np
import torch
import torchvision.transforms as T
from decord import VideoReader, cpu
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

path = "OpenGVLab/InternVL2_5-1B"
checkpoint_path = "./InternVL/internvl_chat/work_dirs/internvl_chat_v2_5/internvl2_5_1b_dynamic_res_2nd_finetune_lora/checkpoint-689"
model = AutoModel.from_pretrained(
    checkpoint_path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True,
    device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)

# set the max number of tiles in `max_num`
pixel_values = load_image("wigner_3d_alpha.png", max_num=12).to(torch.bfloat16)
generation_config = dict(max_new_tokens=1024, do_sample=False)

# pure-text conversation (纯文本对话)
question = 'What is this?'
response, history = model.chat(tokenizer, None, question, generation_config, history=None, return_history=True)
print(f'User: {question}\nAssistant: {response}')


Some weights of the model checkpoint at ./InternVL/internvl_chat/work_dirs/internvl_chat_v2_5/internvl2_5_1b_dynamic_res_2nd_finetune_lora/checkpoint-689 were not used when initializing InternVLChatModel: ['language_model.base_model.model.lm_head.weight', 'language_model.base_model.model.model.embed_tokens.weight', 'language_model.base_model.model.model.layers.0.input_layernorm.weight', 'language_model.base_model.model.model.layers.0.mlp.down_proj.base_layer.weight', 'language_model.base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight', 'language_model.base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight', 'language_model.base_model.model.model.layers.0.mlp.gate_proj.base_layer.weight', 'language_model.base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'language_model.base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'language_model.base_model.model.model.layers.0.mlp.up_proj.base_layer.weight', 'language_model.base_m

User: What is this?
Assistant: !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [1]:
from lmdeploy import pipeline, TurbomindEngineConfig
from lmdeploy.vl import load_image

model = 'OpenGVLab/InternVL2_5-1B'
checkpoint_path = "./InternVL/internvl_chat/work_dirs/internvl_chat_v2_5/internvl2_5_1b_dynamic_res_2nd_finetune_lora/checkpoint-689"

# Add offload_folder here
backend_cfg = TurbomindEngineConfig(session_len=8192, offload_folder="./offload_cache")

pipe = pipeline(checkpoint_path, backend_config=backend_cfg)

prompt = "You are given a grayscale image representing a quantum optical state. Your task is to determine the type of the state (e.g., cat state, Fock state, coherent state, thermal state, etc.) as well as its key parameters (alpha/number of photons/density, number of qubits, and the linear space range). Please provide your answer in the format: \"This is a [STATE TYPE] with [KEY parameters] equal to [VALUE], number of qubits equal to [N] in the linear space [LOW] to [HIGH].\" Then extract your opinion on how you determine state, parameters, number of qubit from the image."

image_urls = ["./wigner_3d_alpha.png"]
prompts = [(prompt, load_image(img_url)) for img_url in image_urls]
response = pipe(prompts)
print(response)


/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


2025-04-18 17:22:13,330 - lmdeploy - WARNING - turbomind.py:214 - get 219 model params


[TM][WARNING] [LlamaTritonModel] `max_context_token_num` is not set, default to 8192.
                                                                   

2025-04-18 17:22:13,369 - lmdeploy - WARNING - turbomind.py:219 - the model may not be loaded successfully with 219 uninitialized params:
['layers.12.ffn_norm.weight', 'layers.14.ffn_norm.weight', 'layers.18.attention.w_qkv.0.bias', 'norm.weight', 'layers.17.attention_norm.weight', 'layers.23.attention.wo.0.bias', 'layers.4.feed_forward.w1.0.weight', 'layers.16.attention.wo.0.weight', 'layers.1.attention.w_qkv.0.bias', 'layers.1.feed_forward.w2.0.weight', 'layers.17.attention.wo.0.weight', 'output.0.weight', 'layers.5.attention.w_qkv.0.weight', 'layers.5.attention.wo.0.bias', 'layers.2.ffn_norm.weight', 'layers.6.feed_forward.w2.0.weight', 'layers.5.attention_norm.weight', 'layers.19.attention.wo.0.weight', 'layers.7.attention.wo.0.bias', 'layers.19.attention.wo.0.bias', 'layers.0.attention.w_qkv.0.weight', 'layers.8.feed_forward.w2.0.weight', 'layers.5.attention.w_qkv.0.bias', 'layers.0.attention.wo.0.bias', 'layers.17.attention.w_qkv.0.weight', 'layers.1.feed_forward.w3.0.weight', 'l

[WARNING] gemm_config.in is not found; using default GEMM algo
2025-04-18 17:22:14,031 - lmdeploy - WARNING - tokenizer.py:499 - The token <|action_end|>, its length of indexes [27, 91, 1311, 6213, 91, 29] is over than 1. Currently, it can not be used as stop words
2025-04-18 17:22:14,127 - lmdeploy - WARNING - async_engine.py:645 - GenerationConfig: GenerationConfig(n=1, max_new_tokens=512, do_sample=False, top_p=1.0, top_k=50, min_p=0.0, temperature=0.8, repetition_penalty=1.0, ignore_eos=False, random_seed=None, stop_words=None, bad_words=None, stop_token_ids=[151643, 151644, 151645], bad_token_ids=None, min_new_tokens=None, skip_special_tokens=True, spaces_between_special_tokens=True, logprobs=None, response_format=None, logits_processors=None, output_logits=None, output_last_hidden_state=None)
2025-04-18 17:22:14,128 - lmdeploy - WARNING - async_engine.py:646 - Since v0.6.0, lmdeploy add `do_sample` in GenerationConfig. It defaults to False, meaning greedy decoding. Please set `do